### Conventions

- $\theta$ is the namedtuple containing all diffsky bounded parameters, i.e., [`ParamCollection`](https://github.com/ArgonneCPAC/diffsky/blob/1ff1b9e47b35daa574f41d4472730a1d2a7f8b9b/diffsky/param_utils/diffsky_param_wrapper_merging.py#L27).

  I refer to it as `param_coll`.

- $\theta^*$ is the namedtuple containing all diffsky unbounded parameters, i.e., [`UParamCollection`](https://github.com/ArgonneCPAC/diffsky/blob/1ff1b9e47b35daa574f41d4472730a1d2a7f8b9b/diffsky/param_utils/diffsky_param_wrapper_merging.py#L46).

  I refer to it as `uparam_coll`.

- $\bar{\theta}$ is the flat version of `ParamCollection`: [`DiffskyParamsFlat`](https://github.com/ArgonneCPAC/diffsky/blob/1ff1b9e47b35daa574f41d4472730a1d2a7f8b9b/diffsky/param_utils/diffsky_param_wrapper_merging.py#L418). This is computed from $\theta$ using `dpwm.unroll_param_collection_into_flat_array`.

  I refer to it as `param_flat`.

- $\bar{\theta}^*$ is the flat version of `UParamCollection`: [`DiffskyUParamsFlat`](https://github.com/ArgonneCPAC/diffsky/blob/1ff1b9e47b35daa574f41d4472730a1d2a7f8b9b/diffsky/param_utils/diffsky_param_wrapper_merging.py#L420). This is computed from $\theta^*$ using `dpwm.unroll_u_param_collection_into_flat_array`.

  I refer to it as `uparam_flat`.

- The subscript $_{\rm var}$ refers to the subset of parameters that we are inferring, i.e., that we are **varying**. We typically compute gradients of the loss/likelihood/log-density defined as a function of **$\bar{\theta}^*_{\rm var}$**, the flat representation of the subset of varied unbounded parameters.

  I refer to $\bar{\theta}^*_{\rm var}$ as `var_uparam_flat` and $\bar{\theta}_{\rm var}$ as `var_param_flat`.

### Basic steps
- load target data
- load diffsky parameters
- get lc_data_phot
- define a likelihood and create ``loss_data``

### HMC steps

0. Check if pre-computed IMM exists: you can compute and save it with the laplace approx. demo notebook
1. Initialize chains
2. Run warmup: get IMM (if not yet provided) and step size estimation
3. Run Sampler
4. Diagnostics

In [ ]:
%cd /home/nvilla/diffsky

# Libraries

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

from diffsky.experimental.lc_generators.lc_phot import mc_lc_phot
from diffsky.param_utils import diffsky_param_wrapper_merging as dpwm
from diffsky.soft_histograms.signdhist_lomem import nnsig_ndhist

from diffsky.experimental.inference import utils, fisher, prior, likelihood, hmc

ran_key = jax.random.key(42)

### Set `dir_out`

To store outputs.

In [ ]:
dir_out = '/home/nvilla/diffstuff_experiments/hmc_dev/scripts_hmc/output'

# Get LC data phot

In [ ]:
from dsps.cosmology import flat_wcdm
from dsps.data_loaders import load_ssp_templates, load_transmission_curve
from diffsky.experimental.lc_generators.lc_data_phot import weighted_lc_data_phot

In [ ]:
# -- Settings --

num_halos = 100
z_min = 0.1
z_max = 0.2
lgmp_min = 10.5
lgmp_max = 15.0
sky_area_degsq = 10

ran_key, lc_data_key = jax.random.split(ran_key, 2)

# Cosmology
cosmo_params = flat_wcdm.PLANCK15
fb = 0.156

n_z_phot_table = 15

# Transmission curves data
filter_names = ["u", "g", "r", "i", "z"]
tcurves_args = {
    "fn": None,
    "bn_pat": "sdss_{}_transmission.h5",
    "drn": "/home/nvilla/diffstuff_data/filters",
}

# SSP data arguments
ssp_data_args = {
    "fn": None,
    "drn": "/home/nvilla/diffstuff_data",
    "bn": "ssp_data_fsps_v3.2_lgmet_age.h5",
}



# -- Get lc_data_phot --

# Load SSP data
ssp_data = load_ssp_templates(**ssp_data_args)

# Load transmission curve data (wavelength array and transmission)
bn_list = [tcurves_args["bn_pat"].format(x) for x in filter_names]
tcurves = [
    load_transmission_curve(
        fn=tcurves_args["fn"], bn_pat=bn, drn=tcurves_args["drn"]
    )
    for bn in bn_list
]

# Define a redshift table used for photometry interpolation
z_phot_table = jnp.linspace(z_min, z_max, n_z_phot_table)

# Get weighted LC data
lc_data_phot = weighted_lc_data_phot(
    lc_data_key,
    num_halos,
    z_min,
    z_max,
    lgmp_min,
    lgmp_max,
    sky_area_degsq=sky_area_degsq,
    ssp_data=ssp_data,
    tcurves=tcurves,
    z_phot_table=z_phot_table,
    cosmo_params=cosmo_params,
    logmp_cutoff=11.0,
)

gal_weight = lc_data_phot.halo_weight

# Load diffsky parameters: initial state

In [ ]:
# Choose param. collection
param_coll = dpwm.DEFAULT_PARAM_COLLECTION

In [ ]:
# Choose varied parameters
var_uparams_list = [
    "u_mean_ulgm_mseq_ytp",
    "u_mean_ulgy_qseq_ytp"
]

var_params_list = [utils.bounded_name(name) for name in var_uparams_list]

In [ ]:
uparam_coll = dpwm.get_u_param_collection_from_param_collection(*param_coll)
uparam_flat = dpwm.unroll_u_param_collection_into_flat_array(*uparam_coll)

var_uparam_flat = utils.get_var_param_flat_from_param_flat(uparam_flat, var_uparams_list)

# List of indices of varied parameters in the flatten namedtuple
var_flat_idx = utils.compute_varied_params_indices(var_uparam_flat, uparam_flat)

# Load target data

In [ ]:
ran_key, target_data_key = jax.random.split(ran_key, 2)
fake_data, _, _ = mc_lc_phot(target_data_key, lc_data_phot, mc_merge=0, param_collection=param_coll)
target_mags = fake_data.obs_mags_weighted

# Define Likelihood

In [ ]:
def target_space_fn(mags):
    return mags[:, 2]


def loglikelihood_from_param_coll(param_coll, loss_data):

    lc_data, XHIST_TARGET, XBINS, loss_key = loss_data

    # Get phot. lightcone
    phot_kern_results, phot_randoms, merging_randoms = mc_lc_phot(
        ran_key,
        lc_data,
        mc_merge=0,
        param_collection=param_coll,
    )
    pred_mags = phot_kern_results.obs_mags_weighted

    # Compute pred. diff. hist
    pred_data = target_space_fn(pred_mags)
    # gal_weight = lc_data.halo_weight
    # gal_weight_masked = get_masked_gal_weight(pred_mags, gal_weight)
    XHIST_PRED = likelihood.soft_xhist(pred_data, XBINS)

    # Compute log-likelihood
    logpdf = likelihood._poisson_kern(XHIST_PRED, XHIST_TARGET)
    
    return logpdf

In [ ]:
target_data = target_space_fn(target_mags)

NBINS = 30
XBOUNDS = (10.0, 40.0)
XBINS = np.linspace(*XBOUNDS, NBINS)[:-1]

XHIST_TARGET = likelihood.soft_xhist(target_data, XBINS)

In [ ]:
ran_key, loss_key = jax.random.split(ran_key, 2)
loss_data = lc_data_phot, XHIST_TARGET, XBINS, loss_key

In [ ]:
XHIST_TARGET, __ = jnp.histogram(target_data, bins=XBINS)
XHIST_TARGET = likelihood.soft_xhist(target_data, XBINS)

fig, ax = plt.subplots(1, 1)
__=ax.plot(XBINS[1:], XHIST_TARGET,
           label='standard histogram')
__=ax.plot(XBINS[1:], XHIST_TARGET,'--',
           label='soft histogram')
leg = ax.legend()

# Define flat partial functions

Gradients will be computed with respect to these functions

In [ ]:
from jax.flatten_util import ravel_pytree

def flat_logposterior_fn(var_uparam_flat, diffsky_params, loss_data, var_flat_idx):
    """
    Combined log-posterior = log-likelihood + log-prior.
    Computes the diffsky transform ``f(*u_coll)`` once and threads the result into both the likelihood (via ``loglikelihood_from_param_coll``) 
    and the prior (soft-uniform term + Jacobian diagonal).
    This fusion avoids the duplicate ``f`` call that would occur if we simply added ``flat_loglikelihood_fn + flat_logprior_fn``.
    """

    # \theta* from \theta*_var
    uparam_coll = utils.get_uparam_coll_from_var_uparam_flat(
        var_uparam_flat, diffsky_params
    )
    # \theta from \theta*
    param_coll = prior.f(*uparam_coll)

    # Likelihood(\theta)
    loglik = loglikelihood_from_param_coll(param_coll, loss_data)

    # -- So far we did the same as flat_loglikelihood_fn. Now we reuse computations to get the prior.

    # Prior: 2 terms
    # first term
    param_flat, _ = ravel_pytree(param_coll)
    lg_dist_term = jnp.sum(
        prior._soft_uniform_log_prior(
            param_flat[var_flat_idx],
            prior.DEFAULT_LOW_FLAT[var_flat_idx],
            prior.DEFAULT_HIGH_FLAT[var_flat_idx],
        )
    )
    # second term
    log_abs_det = prior._var_jac_logdet(uparam_coll, var_flat_idx)

    return loglik + lg_dist_term + log_abs_det
    

In [ ]:
from functools import partial

flat_logposterior = partial(
    flat_logposterior_fn,
    diffsky_params=uparam_flat,
    loss_data=loss_data,
    var_flat_idx=var_flat_idx,
)

# HMC

- initialize chains
- run warmup: compute IMM and step size
- run sampler
- compute diagnostics

First, let's load a precomputed IMM, if there exists any.

It will be used to run the chain initialization, and also to run the sampler.

If the IMM is not provided, it's no big deal for chain initilization, but the warmup will compute the IMM in addition to the step size, which may take too long. 

In [ ]:
import os

imm_path = f"{dir_out}/covariance_matrix.npy"
if os.path.exists(imm_path):
    imm = np.load(imm_path)
    print(f"loaded precomputed IMM from {imm_path}")
else:
    # will compute IMM with warmup
    imm = None

## Chain initialization

You can run multiple chains in parallel with the existing code.

In [ ]:
n_devices = jax.local_device_count()
print(n_devices)

In [ ]:
# Settings
num_samples = 100
warmup_num_steps = 100
max_num_doublings = 10
target_accept = 0.8
initial_step_size = 1.0
num_chains = 2  # n_devices
init_jitter = 2

hmc_settings = {
    'warmup_num_steps': warmup_num_steps,
    'max_num_doublings': max_num_doublings,
    'target_accept': target_accept,
    'initial_step_size': initial_step_size,
}

# Random keys
warmup_key, sampler_key, init_key = jax.random.split(ran_key, 3)
warmup_keys = jax.random.split(warmup_key, num_chains)
sampler_keys = jax.random.split(sampler_key, num_chains)

In [ ]:
# Run chain initialization
chain_inits = hmc.make_chain_inits(
        var_uparam_flat,
        num_chains,
        init_jitter,
        imm,
        init_key,
    )

In [ ]:
chain_inits

## Warmup

The number of warmup steps and the number of samples should be typically much larger than what is done here.
This is just for demonstrtation

In [ ]:
warmup_states, step_sizes, warmup_info, imm_out = hmc.run_warmup(
    flat_logposterior,
    warmup_keys,
    chain_inits,
    hmc_settings,
    inverse_mass_matrix=imm,
)

## Run sampler

In [ ]:
positions, sample_info = hmc.run_sampling(
    flat_logposterior,
    imm_out,
    max_num_doublings,
    num_samples,
    sampler_keys,
    warmup_states,
    step_sizes,
)

# Save samples

The next cell saves the samples into a single namedtuple.

If you run multiple chains, there are functions to save the samples from each chain separately as well.

In [ ]:
positions_concat = jax.tree_util.tree_map(lambda x: x.reshape(-1), positions)
uparam_flat_samples = uparam_flat._replace(**positions_concat._asdict())

# get uniform depth pytree
uparam_flat_samples = utils.get_flat_params_all_same_shape(uparam_flat_samples)
# convert into collection
uparam_coll_samples = dpwm.get_u_param_collection_from_u_param_array(uparam_flat_samples)
# convert into bounded
param_coll_samples = dpwm.get_param_collection_from_u_param_collection(*uparam_coll_samples)


# Save ParamCollection with samples
# from diffsky.data_loaders.hacc_utils.lc_mock import write_diffsky_param_collection_merging
# write_diffsky_param_collection_merging(dir_out, "fisher_samples", param_coll)

## Diagnostics

### Visualize samples

In [ ]:
# from diffsky.data_loaders.hacc_utils.lc_mock import load_diffsky_param_collection_merging
# param_coll_samples = load_diffsky_param_collection_merging(dir_out, "fisher_samples")

param_flat_samples = dpwm.unroll_param_collection_into_flat_array(*param_coll_samples)

In [ ]:
default_flat = dpwm.unroll_param_collection_into_flat_array(*dpwm.DEFAULT_PARAM_COLLECTION)

plt.scatter(param_flat_samples.mean_ulgm_mseq_ytp, param_flat_samples.mean_ulgy_qseq_ytp, s=1)
plt.axvline(default_flat.mean_ulgm_mseq_ytp, color='k', ls='--')
plt.axhline(default_flat.mean_ulgy_qseq_ytp, color='k', ls='--')
plt.show()

### ppd

In [ ]:
def get_photometry(param_coll, lc_data):
    
    phot_kern_results, phot_randoms, merging_randoms = mc_lc_phot(
        ran_key,
        lc_data,
        mc_merge=0,
        param_collection=param_coll,
    )

    return phot_kern_results.obs_mags_weighted

In [ ]:
individual_param_coll = utils.unpack_nested_samples(param_coll_samples)

pred_mags_list = []
diffstarpop_params_list = []
for sample_ind in range(100):
    single_param_coll = individual_param_coll[sample_ind]
    pred_mags = get_photometry(single_param_coll, lc_data_phot)

    pred_mags_list.append(pred_mags)
    diffstarpop_params_list.append(single_param_coll.diffstarpop_params)

In [ ]:
plt.figure(figsize=(4, 4))

for m in pred_mags_list:
    m_hist, _ = np.histogram(m, bins=XBINS)
    plt.plot(m_hist, color='gray', alpha=0.25)

plt.show()

In [ ]:
from diffsky.experimental.diagnostics.check_smhm import plot_diffstarpop_insitu_smhm_multi
plot_diffstarpop_insitu_smhm_multi(diffstarpop_params_list)

### Multi-chain diagnostics

In [ ]:
from blackjax.diagnostics import potential_scale_reduction, effective_sample_size

positions_flat = jax.vmap(jax.vmap(jnp.asarray))(positions)

print(potential_scale_reduction(positions_flat))
print(effective_sample_size(positions_flat))

In [ ]:
for p in range(len(var_uparams_list)):
    plt.figure(figsize=(5, 3))
    for c in range(num_chains):
        plt.plot(positions_flat[c, :, p], label=f'chain {c}')

    plt.ylabel(f'{var_uparams_list[p]}')
    plt.xlabel('sample')

    plt.legend()
    plt.show()